In [4]:
# Make necessary imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn libraries for datasets
from sklearn.datasets import make_moons
from sklearn.datasets import make_moons
from sklearn.datasets import make_blobs
from sklearn.datasets import make_circles

# Sklearn libraries for ml models
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans

# Sklearn libraries for metrics
from sklearn.metrics import silhouette_score


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Creating a dataset to make our models learn on
X, y = make_moons(
    n_samples=1000,
    noise=0.10,
    random_state=42
    )

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 5 samples:\n", X[:5])

In [ ]:
# Visualize the dataset to get a better feel for it
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="rainbow", s = 10)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Moon Dataset")
plt.grid(True)
plt.show()

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

# We will wrap the model instantiation and plotting in a single function
def interactive_dbscan(epsilon, min_samples):
    # Fit the DBSCAN model
    dbscan = DBSCAN(eps=epsilon, min_samples=min_samples, metric="euclidean")
    labels = dbscan.fit_predict(X)

    # Plotting
    plt.figure(figsize=(10, 6), dpi=100)

    # Scatter points. Noise points are labeled as -1, which we'll make stand out.
    scatter = plt.scatter(
        X[:, 0], X[:, 1],
        c=labels, cmap="plasma",
        s=40, edgecolors="k", linewidths=0.5, alpha=0.8
    )

    plt.title(f"Interactive DBSCAN: eps={epsilon:.2f}, min_samples={min_samples}", fontsize=14, fontweight="bold")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.show()

# Generate the interactive sliders
interact(
    interactive_dbscan,
    epsilon=widgets.FloatSlider(value=0.1, min=0.01, max=0.3, step=0.01, description='Epsilon:'),
    min_samples=widgets.IntSlider(value=5, min=1, max=20, step=1, description='Min Samples:')
)

In [ ]:
def get_neighbors(data, point_idx, epsilon):
    """
    Finds all points within `epsilon` distance from `data[point_idx]`.
    """
    neighbors = []

    for i in range(len(data)):
        # TODO 1: Calculate the Euclidean distance between data[point_idx] and data[i]
        # HINT: np.linalg.norm() calculates the magnitude of a vector
        distance = np.linalg.norm(data[point_idx] - data[i])

        # TODO 2: If the distance is less than or equal to epsilon, it's a neighbor!
        if distance <= epsilon:
            neighbors.append(i)

    return neighbors

In [ ]:
def custom_dbscan(data, epsilon, min_samples):
    labels = np.zeros(len(data), dtype=int) # Start with all 0s (Unvisited)
    cluster_id = 0

    for p in range(len(data)):
        # Skip if we've already visited this point
        if labels[p] != 0:
            continue

        # Get neighbors using our helper function
        neighbors = get_neighbors(data, p, epsilon)

        # TODO 3: Check if the current point is a CORE point
        if len(neighbors) < min_samples:
            # Not enough neighbors? Label it as noise for now.
            labels[p] = -1
        else:
            # We found a core point! Start a new cluster.
            cluster_id += 1
            labels[p] = cluster_id

            # --- BFS Queue Expansion ---
            # We use a while loop because 'neighbors' will grow as we find more points
            i = 0
            while i < len(neighbors):
                neighbor_idx = neighbors[i]

                # TODO 4: If this neighbor was previously labeled NOISE, claim it for our current cluster
                if labels[neighbor_idx] == -1:
                    labels[neighbor_idx] = cluster_id

                # TODO 5: If this neighbor is completely UNVISITED, process it
                elif labels[neighbor_idx] == 0:
                    # Assign it to the current cluster
                    labels[neighbor_idx] = cluster_id

                    # Check if THIS neighbor is ALSO a core point
                    new_neighbors = get_neighbors(data, neighbor_idx, epsilon)

                    # TODO 6: If it has enough neighbors, add them to our BFS queue to keep expanding!
                    if len(new_neighbors) >= min_samples:
                        neighbors.extend(new_neighbors)

                i += 1 # Move to the next item in the queue

    return labels

In [ ]:
# Tune the following hyperparameters

# Hyperparameters for DBSCAN
EPSILON = 0.1
MIN_SAMPLES = 5

In [ ]:
# Let's test your implementation!
# (Ensure your Epsilon and min_samples match the ones we tuned earlier)
my_labels = custom_dbscan(X, epsilon=EPSILON, min_samples=MIN_SAMPLES)

plt.figure(figsize=(8, 5))
plt.scatter(
    X[:, 0], X[:, 1],
    c=my_labels, cmap="plasma",
    s=40, edgecolors="k", linewidths=0.5
)
plt.title("My Custom DBSCAN Output", fontsize=14, fontweight="bold")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

In [ ]:
my_labels = custom_dbscan(X, epsilon=0.12, min_samples=6)

# 2. Check how many unique clusters your code actually found
unique_labels = np.unique(my_labels)
print(f"Clusters found by your DIY algorithm (Note: -1 is noise): {unique_labels}")

# 3. Calculate the Silhouette Score safely
if len(unique_labels) > 1:
    diy_score = silhouette_score(X, my_labels)
    print(f"🏆 Your DIY DBSCAN Silhouette Score: {diy_score:.4f}")

else:
    print("Oops! Your DIY algorithm only found 1 cluster (or just noise).")
    print("Silhouette score requires at least 2 clusters to calculate.")
    print("Check your BFS queue expansion logic!")